# Stich Great Lake DEM

In [ ]:
import os, glob
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling

Modeltop = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM"
target_crs = "EPSG:3174"
target_res = 30
nodata_out = -9999.0

tmp_dir = os.path.join(Modeltop, "_tmp_reproj_3174")
os.makedirs(tmp_dir, exist_ok=True)

tifs = sorted(glob.glob(os.path.join(Modeltop, "*.tif")))
if not tifs:
    raise FileNotFoundError(f"No .tif files found in: {Modeltop}")

reproj_paths = []

for fp in tifs:
    with rasterio.open(fp) as src:
        if src.crs is None:
            raise ValueError(f"{os.path.basename(fp)} has no CRS. Define Projection first.")

        src_nodata = src.nodata
        out_fp = os.path.join(tmp_dir, os.path.splitext(os.path.basename(fp))[0] + "_3174.tif")

        if not os.path.exists(out_fp):
            transform, width, height = calculate_default_transform(
                src.crs, target_crs, src.width, src.height, *src.bounds
            )

            meta = src.meta.copy()
            meta.update({
                "crs": target_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "dtype": "float32",
                "nodata": nodata_out,
                "compress": "lzw",
                "tiled": True,
                "BIGTIFF": "IF_SAFER"
            })

            with rasterio.open(out_fp, "w", **meta) as dst:
                reproject(
                    source=rasterio.band(src, 1),
                    destination=rasterio.band(dst, 1),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    src_nodata=src_nodata,
                    dst_transform=transform,
                    dst_crs=target_crs,
                    dst_nodata=nodata_out,
                    resampling=Resampling.bilinear
                )

        reproj_paths.append(out_fp)

# Merge and write directly to disk (BigTIFF)
out_tif = os.path.join(Modeltop, "DEM_merged_3174_30m.tif")

srcs = [rasterio.open(fp) for fp in reproj_paths]
template = srcs[0].meta.copy()
template.update({
    "driver": "GTiff",
    "crs": target_crs,
    "dtype": "float32",
    "nodata": nodata_out,
    "compress": "lzw",
    "tiled": True,
    "BIGTIFF": "YES"
})

merge(
    srcs,
    res=(target_res, target_res),
    nodata=nodata_out,
    resampling=Resampling.bilinear,
    target_aligned_pixels=True,
    dst_path=out_tif,
    dst_kwds=template
)

for s in srcs:
    s.close()

print("✅ Wrote:", out_tif)


In [1]:
import arcpy
arcpy.env.overwriteOutput = True

out_tif = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

arcpy.management.CalculateStatistics(out_tif)
arcpy.management.BuildPyramids(out_tif)
print("✅ Built statistics + pyramids")

✅ Built statistics + pyramids


# Stich the Lakes

In [ ]:
out_dir  = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers"


In [ ]:
import os
import arcpy

Lakes_path = r"D:\Users\abolmaal\Arcgis\GreatLakes"
out_shp = os.path.join(Lakes_path, "hydro_p_GreatLakes.shp")

arcpy.env.workspace = Lakes_path
arcpy.env.overwriteOutput = True

# Get all shapefiles in the folder
shps = arcpy.ListFeatureClasses(feature_type="All")
shps = [os.path.join(Lakes_path, s) for s in shps if s.lower().endswith(".shp")]

if not shps:
    raise FileNotFoundError(f"No shapefiles found in: {Lakes_path}")

# Merge
arcpy.management.Merge(shps, out_shp)

print(f"✅ Merged {len(shps)} shapefiles -> {out_shp}")


✅ Merged 5 shapefiles -> D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp


# Create constant heads boundary 


to make a constant-head (CHD) boundary from your basin DEM, you basically do two things:

1- decide which grid cells are “constant head” (usually the Great Lakes water cells, and sometimes the outer model boundary), and

2- assign a head value (stage elevation) to those cells for each stress period.

### 1) Rasterize your hydro polygons to your 30 m model grid (aligned to your DEM)

This makes a water_30m.tif where water cell = -1, outside = 0, inside (land) = 1

In [6]:
import os
import arcpy
from arcpy import sa

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ----------------------------
# INPUTS
# ----------------------------
hydro_p = r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"
dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"
boundary_p = r"S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp"

out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads"
os.makedirs(out_dir, exist_ok=True)

buffer_dist_m = 20000  # 20 km buffer

# ----------------------------
# ENV ALIGNMENT
# ----------------------------
arcpy.env.snapRaster = dem
arcpy.env.cellSize   = dem
arcpy.env.extent     = dem
arcpy.env.outputCoordinateSystem = arcpy.Describe(dem).spatialReference

dem_r  = arcpy.Raster(dem)
dem_sr = arcpy.Describe(dem).spatialReference

print("DEM rows/cols:", dem_r.height, dem_r.width)
print("DEM cellsize:", dem_r.meanCellWidth, dem_r.meanCellHeight)
print("DEM SR:", dem_sr.name, "factoryCode:", dem_sr.factoryCode)

# ----------------------------
# HELPERS
# ----------------------------
def safe_delete(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)

def project_if_needed(fc, out_name):
    sr = arcpy.Describe(fc).spatialReference
    if sr.factoryCode != dem_sr.factoryCode:
        out_fc = os.path.join(out_dir, out_name)
        safe_delete(out_fc)
        arcpy.management.Project(fc, out_fc, dem_sr)
        return out_fc
    return fc

def ensure_constant_field(fc, field_name, value=1):
    fset = {f.name.upper() for f in arcpy.ListFields(fc)}
    if field_name.upper() not in fset:
        arcpy.management.AddField(fc, field_name, "SHORT")
    arcpy.management.CalculateField(fc, field_name, str(value), "PYTHON3")
    return field_name

# ----------------------------
# PROJECT INPUTS
# ----------------------------
hydro_use = project_if_needed(hydro_p, "hydro_proj.shp")
bnd_use   = project_if_needed(boundary_p, "boundary_proj.shp")

# ----------------------------
# FIX GEOMETRY (optional but helps with weird gaps)
# ----------------------------
try:
    arcpy.management.RepairGeometry(bnd_use)
except Exception as e:
    print("⚠️ RepairGeometry skipped/failed:", e)

# ----------------------------
# 2 KM BUFFER AROUND BOUNDARY
# ----------------------------
bnd_buff = os.path.join(out_dir, f"boundary_buffer_{buffer_dist_m}m.shp")
safe_delete(bnd_buff)

# dissolve_option="ALL" prevents internal seams
arcpy.analysis.Buffer(
    in_features=bnd_use,
    out_feature_class=bnd_buff,
    buffer_distance_or_field=f"{buffer_dist_m} Meters",
    dissolve_option="ALL"
)

print("✅ Buffered boundary saved:", bnd_buff)

# ----------------------------
# RASTERIZE BUFFERED BOUNDARY: inside=1, outside=0
# ----------------------------
bnd_field = ensure_constant_field(bnd_buff, "BNDVAL", 1)

bnd_raw = os.path.join(out_dir, "boundary_buff_raw.tif")
safe_delete(bnd_raw)

arcpy.conversion.PolygonToRaster(
    in_features=bnd_buff,
    value_field=bnd_field,
    out_rasterdataset=bnd_raw,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)

bnd_ras = sa.Int(sa.Con(sa.IsNull(bnd_raw), 0, 1))  # 0/1 everywhere

# ----------------------------
# RASTERIZE WATER: water=1, else 0
# ----------------------------
wat_field = ensure_constant_field(hydro_use, "WATVAL", 1)

wat_raw = os.path.join(out_dir, "water_raw.tif")
safe_delete(wat_raw)

arcpy.conversion.PolygonToRaster(
    in_features=hydro_use,
    value_field=wat_field,
    out_rasterdataset=wat_raw,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)

water_ras = sa.Int(sa.Con(sa.IsNull(wat_raw), 0, 1))

# ----------------------------
# FINAL MASK:
# outside buffered boundary = 0
# inside buffered boundary (non-water) = 1
# water = -1  (only within buffered boundary)
# ----------------------------
base  = bnd_ras
final = sa.Con(water_ras == 1, -1, base)
final = sa.Con(bnd_ras == 0, 0, final)   # enforce outside=0
final = sa.Int(final)

# ----------------------------
# SAVE TO FILE GEODATABASE
# ----------------------------
gdb = os.path.join(out_dir, "rasters.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, "rasters.gdb")

out_gdb_ras = os.path.join(gdb, f"domain_water_mask_30m_buff{buffer_dist_m}m")
safe_delete(out_gdb_ras)

arcpy.management.CopyRaster(
    in_raster=final,
    out_rasterdataset=out_gdb_ras,
    pixel_type="16_BIT_SIGNED",
    nodata_value="0"
)

print("✅ Saved to GDB raster:", out_gdb_ras)

# ----------------------------
# OPTIONAL: export to TIFF (may fail if huge)
# ----------------------------
out_tif = os.path.join(out_dir, f"domain_water_mask_30m_buff{buffer_dist_m}m.tif")
safe_delete(out_tif)

try:
    arcpy.management.CopyRaster(
        in_raster=out_gdb_ras,
        out_rasterdataset=out_tif,
        pixel_type="16_BIT_SIGNED",
        nodata_value="0",
        format="TIFF"
    )
    print("✅ Exported TIFF:", out_tif)
except Exception as e:
    print("⚠️ TIFF export failed (likely too large). Keep the GDB raster.")
    print("   Error:", e)

# ----------------------------
# SMALL SAMPLE CHECK
# ----------------------------
import numpy as np
sample = arcpy.RasterToNumPyArray(out_gdb_ras, ncols=500, nrows=500, nodata_to_value=999)
vals = np.unique(sample[sample != 999])
print("✅ sample unique values:", vals)


DEM rows/cols: 49758 66126
DEM cellsize: 30.0 30.0
DEM SR: NAD_1983_Great_Lakes_Basin_Albers factoryCode: 3174
✅ Buffered boundary saved: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_20000m.shp
✅ Saved to GDB raster: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\rasters.gdb\domain_water_mask_30m_buff20000m
✅ Exported TIFF: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif
✅ sample unique values: []


# Making HK
Below is a single ArcPy script that does everything end-to-end:

1- Clip surface geology to extended_Buffer2km

2- Add UPKh_m_d, MidKh_m_d, LowKh_m_d

3-Set those = 100 where intersecting hydro_p

4- Convert Excel → table (inside a File GDB)

5- Create robust text join keys (avoids numeric/text mismatch)

6- Join Excel fields onto the final output

7- Export a final feature class (and optional shapefile)

Important note: Joining Excel can add many fields and shapefiles have field name limits (10 chars). I output to a File Geodatabase feature class by default (recommended). You can still export a shapefile if you want.


# ------------------------------------------------------------
### INPUTS
# ------------------------------------------------------------
surfacegeology = r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp"
extended_Buffer2km = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_2000m.shp"
hydro_p = r"D:\Users\abolmaal\Arcgis\GreatLakes\hydro_p_GreatLakes.shp"

excel_xlsx = r"S:\Data\Other_Data\Xu_2021\Data\11_Quaternary_Material_Description_Calibrated_Kh\Calibrated_surficial_Kh.xlsx"
EXCEL_SHEET_NAME = "GLB_surf_dissolve_merged"


In [22]:
import os
import arcpy
from arcpy.sa import Raster, SetNull

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ============================================================
# INPUTS
# ============================================================
surfacegeology = r"S:\Data\Other_Data\Xu_2021\Data\09_Integrated_Surficial_Geology_Map\Integrated_surficial_geology_mapping.shp"
csv_path = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\Calibrated_surficial_Kh__ALLCOLS.csv"

# Domain polygon (your 2 km buffer)
extended_Buffer2km = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\boundary_buffer_2000m.shp"

# IMPORTANT: use the raster mask as truth for lakes
water_mask_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif"
LAKE_RASTER_VALUE = -1

LOWER_DEFAULT = 0.086

# What to assign in lake areas
LAKE_VALUE = 100.0
LAND_VALUE = 0.0

# If you want lakes to also have HK=100 (recommended for CHD zones)
SET_HK_TO_LAKE_VALUE = True

# Optional: remove other numeric attributes from lake polygons
CLEAR_OTHER_NUMERIC_ON_LAKE = True

# ============================================================
# OUTPUTS
# ============================================================
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK"
os.makedirs(out_dir, exist_ok=True)

gdb = os.path.join(out_dir, "HK_join_clip.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(gdb))

surf_fc     = os.path.join(gdb, "surfacegeology_base")
csv_tbl     = os.path.join(gdb, "Calibrated_surficial_Kh_tbl")
joined_fc   = os.path.join(gdb, "surfacegeo_joined")

# persistent projected domain polygon (FIX for your tmp_buf delete problem)
domain_fc   = os.path.join(gdb, "domain_buffer2km_proj")

clip_fc     = os.path.join(gdb, "surfacegeo_joined_clip2km")

# lake polygon from raster
lake_ras    = os.path.join(gdb, "lake_mask01_r")
lake_poly0  = os.path.join(gdb, "lake_poly_raw")
lake_poly   = os.path.join(gdb, "lake_poly_diss")
lake_poly_hk= os.path.join(gdb, "lake_poly_hkCRS")
lake_domain = os.path.join(gdb, "lake_in_domain")

# split/fill outputs
lake_part    = os.path.join(gdb, "hk_lake_part")
land_part    = os.path.join(gdb, "hk_land_part")
lake_missing = os.path.join(gdb, "hk_lake_missing")

final_fc     = os.path.join(gdb, "surfacegeo_HK_FINAL_withLakeFill")
final_shp    = os.path.join(out_dir, "surfacegeology_Kh_clip2km_Rech.shp")

# ============================================================
# HELPERS
# ============================================================
def delete_if_exists(p):
    if p and arcpy.Exists(p):
        arcpy.management.Delete(p)

def same_sr(sr1, sr2):
    try:
        if sr1.factoryCode and sr2.factoryCode:
            return sr1.factoryCode == sr2.factoryCode
    except Exception:
        pass
    return (sr1.name or "") == (sr2.name or "")

def ensure_field(dataset, name, ftype, length=None):
    existing = {f.name for f in arcpy.ListFields(dataset)}
    if name not in existing:
        if ftype.upper() == "TEXT" and length is not None:
            arcpy.management.AddField(dataset, name, ftype, field_length=length)
        else:
            arcpy.management.AddField(dataset, name, ftype)

def to_key_py(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "":
        return None
    try:
        fv = float(s)
        if fv.is_integer():
            s = str(int(fv))
        else:
            s = str(fv)
    except Exception:
        pass
    return s.strip().upper()

def make_join_id(dataset, source_field, join_field="JOIN_ID"):
    ensure_field(dataset, join_field, "TEXT", length=50)
    codeblock = r"""
def to_key(v):
    if v is None:
        return None
    s = str(v).strip()
    if s == "":
        return None
    try:
        fv = float(s)
        if fv.is_integer():
            s = str(int(fv))
        else:
            s = str(fv)
    except:
        pass
    return s.strip().upper()
"""
    arcpy.management.CalculateField(
        dataset, join_field,
        expression=f"to_key(!{source_field}!)",
        expression_type="PYTHON3",
        code_block=codeblock
    )

def count_intersections(fc, fc_field, tbl, tbl_field):
    fc_keys = set()
    with arcpy.da.SearchCursor(fc, [fc_field]) as cur:
        for (v,) in cur:
            k = to_key_py(v)
            if k is not None:
                fc_keys.add(k)

    tb_keys = set()
    with arcpy.da.SearchCursor(tbl, [tbl_field]) as cur:
        for (v,) in cur:
            k = to_key_py(v)
            if k is not None:
                tb_keys.add(k)

    return len(fc_keys.intersection(tb_keys)), len(fc_keys), len(tb_keys)

def safe_clip(in_fc, clip_fc, out_fc):
    """Prefer PairwiseClip if available, else Clip."""
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseClip"):
        arcpy.analysis.PairwiseClip(in_fc, clip_fc, out_fc)
    else:
        arcpy.analysis.Clip(in_fc, clip_fc, out_fc)

def safe_erase(in_fc, erase_fc, out_fc):
    """Prefer PairwiseErase if available, else Erase."""
    delete_if_exists(out_fc)
    if hasattr(arcpy.analysis, "PairwiseErase"):
        arcpy.analysis.PairwiseErase(in_fc, erase_fc, out_fc)
    else:
        arcpy.analysis.Erase(in_fc, erase_fc, out_fc)

def clear_editable_numeric_fields(fc, keep_fields):
    numeric_types = {"Double", "Single", "Integer", "SmallInteger"}
    fields_to_clear = []
    for fld in arcpy.ListFields(fc):
        nm = fld.name
        if fld.type not in numeric_types:
            continue
        if nm in keep_fields:
            continue
        if nm.lower() in ("shape_area", "shape_length", "shape_leng"):
            continue
        if getattr(fld, "required", False):
            continue
        if hasattr(fld, "editable") and (fld.editable is False):
            continue
        fields_to_clear.append(nm)

    for nm in fields_to_clear:
        arcpy.management.CalculateField(fc, nm, "None", "PYTHON3")

# ============================================================
# 1) Copy surfacegeology to GDB
# ============================================================
print("1/8 Copy surfacegeology to GDB...")
delete_if_exists(surf_fc)
arcpy.management.CopyFeatures(surfacegeology, surf_fc)

surf_fields = [f.name for f in arcpy.ListFields(surf_fc)]
if "POLYID" not in surf_fields:
    raise RuntimeError(f"POLYID not found in surfacegeology. Fields: {surf_fields}")
print("  ✅", surf_fc)

# ============================================================
# 2) Import CSV to GDB table
# ============================================================
print("2/8 Import CSV -> GDB table...")
delete_if_exists(csv_tbl)
arcpy.conversion.TableToTable(csv_path, gdb, os.path.basename(csv_tbl))
print("  ✅", csv_tbl)

tbl_fields = [f.name for f in arcpy.ListFields(csv_tbl)]
print("  CSV fields found:", tbl_fields)

if "Upper_ms" not in tbl_fields or "Middle_ms" not in tbl_fields:
    raise RuntimeError(f"CSV table must contain Upper_ms and Middle_ms.\nFields found: {tbl_fields}")

if "Lower_ms" not in tbl_fields:
    print(f"  ⚠️ Lower_ms not found. Creating Lower_ms = {LOWER_DEFAULT} ...")
    ensure_field(csv_tbl, "Lower_ms", "DOUBLE")
    arcpy.management.CalculateField(csv_tbl, "Lower_ms", str(LOWER_DEFAULT), "PYTHON3")
    tbl_fields = [f.name for f in arcpy.ListFields(csv_tbl)]

candidates = [c for c in ["col_1", "col_2"] if c in tbl_fields]
if not candidates:
    raise RuntimeError(f"CSV table missing col_1/col_2. Fields found: {tbl_fields}")

# ============================================================
# 3) Determine join key
# ============================================================
print("3/8 Determine join key (POLYID <-> col_1 or col_2)...")
best_field, best_match = None, -1
debug = []
for c in candidates:
    m, ns, nt = count_intersections(surf_fc, "POLYID", csv_tbl, c)
    debug.append((c, m, ns, nt))
    if m > best_match:
        best_match = m
        best_field = c

print("  Overlap tests (candidate, matches, surf_unique, table_unique):")
for row in debug:
    print("   ", row)

if best_match <= 0:
    raise RuntimeError("No matches found between surface POLYID and CSV col_1/col_2.")
print(f"  ✅ Using CSV join field: {best_field} (matches={best_match})")

# ============================================================
# 4) Join CSV -> surfacegeology
# ============================================================
print("4/8 Join CSV -> surfacegeology...")
make_join_id(surf_fc, "POLYID", "JOIN_ID")
make_join_id(csv_tbl, best_field, "JOIN_ID")

delete_if_exists(joined_fc)
arcpy.management.CopyFeatures(surf_fc, joined_fc)

csv_fields_to_add = []
for f in arcpy.ListFields(csv_tbl):
    if f.type in ("OID", "Geometry"):
        continue
    if f.name.lower() == "join_id":
        continue
    csv_fields_to_add.append(f.name)

arcpy.management.JoinField(joined_fc, "JOIN_ID", csv_tbl, "JOIN_ID", csv_fields_to_add)
print(f"  ✅ Joined {len(csv_fields_to_add)} CSV columns.")

# ============================================================
# 5) Create HK fields from Upper/Middle/Lower
# ============================================================
print("5/8 Create UPKh_m_d, MidKh_m_d, LowKh_m_d...")

for fld in ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]:
    ensure_field(joined_fc, fld, "DOUBLE")

codeblock = r"""
def to_float(v):
    if v is None:
        return None
    try:
        s = str(v).strip()
        if s == "":
            return None
        return float(s)
    except:
        return None
"""
arcpy.management.CalculateField(joined_fc, "UPKh_m_d",  "to_float(!Upper_ms!)",  "PYTHON3", codeblock)
arcpy.management.CalculateField(joined_fc, "MidKh_m_d", "to_float(!Middle_ms!)", "PYTHON3", codeblock)
arcpy.management.CalculateField(joined_fc, "LowKh_m_d", "to_float(!Lower_ms!)",  "PYTHON3", codeblock)

# ============================================================
# 6) Project domain polygon ONCE (persistently) + Clip geology
# ============================================================
print("6/8 Clip to 2 km buffer polygon (persistent projected domain)...")

fc_sr  = arcpy.Describe(joined_fc).spatialReference
buf_sr = arcpy.Describe(extended_Buffer2km).spatialReference

delete_if_exists(domain_fc)
if same_sr(fc_sr, buf_sr):
    arcpy.management.CopyFeatures(extended_Buffer2km, domain_fc)
else:
    arcpy.management.Project(extended_Buffer2km, domain_fc, fc_sr)

arcpy.management.RepairGeometry(domain_fc)

delete_if_exists(clip_fc)
safe_clip(joined_fc, domain_fc, clip_fc)
arcpy.management.RepairGeometry(clip_fc)
print("  ✅", clip_fc)

# ============================================================
# 7) Split by lake raster (-1) + fill missing lake polygons
# ============================================================
print("7/8 Split by lake raster (-1) + fill missing lake areas...")

# Ensure Rech_m_d exists
ensure_field(clip_fc, "Rech_m_d", "DOUBLE")

# Build lake-only raster (1 where water mask == -1, else NoData)
delete_if_exists(lake_ras)
ras = Raster(water_mask_raster)
lake_only = SetNull(ras != LAKE_RASTER_VALUE, 1)
lake_only.save(lake_ras)

# Raster to polygon, dissolve
delete_if_exists(lake_poly0)
delete_if_exists(lake_poly)
arcpy.conversion.RasterToPolygon(lake_ras, lake_poly0, "SIMPLIFY", "VALUE")
arcpy.management.Dissolve(lake_poly0, lake_poly)
arcpy.management.RepairGeometry(lake_poly)

# Project lake polygon to HK CRS
lake_sr = arcpy.Describe(lake_poly).spatialReference
delete_if_exists(lake_poly_hk)
if same_sr(lake_sr, fc_sr):
    arcpy.management.CopyFeatures(lake_poly, lake_poly_hk)
else:
    arcpy.management.Project(lake_poly, lake_poly_hk, fc_sr)
arcpy.management.RepairGeometry(lake_poly_hk)

# Clip lake polygon to domain
delete_if_exists(lake_domain)
safe_clip(lake_poly_hk, domain_fc, lake_domain)
arcpy.management.RepairGeometry(lake_domain)

# Split geology into lake-part and land-part
delete_if_exists(lake_part)
delete_if_exists(land_part)
safe_clip(clip_fc, lake_domain, lake_part)
safe_erase(clip_fc, lake_domain, land_part)

# Find missing lake area not covered by geology polygons
delete_if_exists(lake_missing)
# (lake_missing gets only lake geometry; no geology attrs)
safe_erase(lake_domain, lake_part, lake_missing)

print("  lake_part count   =", int(arcpy.management.GetCount(lake_part)[0]))
print("  land_part count   =", int(arcpy.management.GetCount(land_part)[0]))
print("  lake_missing count=", int(arcpy.management.GetCount(lake_missing)[0]))

# Make sure lake_missing has HK + Rech fields (so merge schema includes them)
ensure_field(lake_missing, "Rech_m_d", "DOUBLE")
for f in ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]:
    ensure_field(lake_missing, f, "DOUBLE")

# LAND: set Rech=0
arcpy.management.CalculateField(land_part, "Rech_m_d", str(LAND_VALUE), "PYTHON3")

# LAKE polygons (both existing lake_part and lake_missing): set Rech=100
arcpy.management.CalculateField(lake_part, "Rech_m_d", str(LAKE_VALUE), "PYTHON3")
arcpy.management.CalculateField(lake_missing, "Rech_m_d", str(LAKE_VALUE), "PYTHON3")

# Set HK in lakes
if SET_HK_TO_LAKE_VALUE:
    for f in ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]:
        arcpy.management.CalculateField(lake_part, f, str(LAKE_VALUE), "PYTHON3")
        arcpy.management.CalculateField(lake_missing, f, str(LAKE_VALUE), "PYTHON3")

# Optional: clear other numeric fields on lake polygons so they don't carry geology attrs
if CLEAR_OTHER_NUMERIC_ON_LAKE:
    keep = set(["Rech_m_d", "UPKh_m_d", "MidKh_m_d", "LowKh_m_d"])
    clear_editable_numeric_fields(lake_part, keep)
    # lake_missing already lacks most fields, but safe anyway:
    clear_editable_numeric_fields(lake_missing, keep)

# Merge all back
delete_if_exists(final_fc)
arcpy.management.Merge([land_part, lake_part, lake_missing], final_fc)
arcpy.management.RepairGeometry(final_fc)

print("  ✅ Final merged FC:", final_fc)
print("     total count =", int(arcpy.management.GetCount(final_fc)[0]))

# ============================================================
# 8) Export final shapefile
# ============================================================
print("8/8 Export final shapefile...")

# Drop helper field if present
if "JOIN_ID" in [f.name for f in arcpy.ListFields(final_fc)]:
    try:
        arcpy.management.DeleteField(final_fc, ["JOIN_ID"])
    except:
        pass

delete_if_exists(final_shp)
arcpy.conversion.FeatureClassToFeatureClass(final_fc, out_dir, os.path.basename(final_shp))

print("\nDONE ✅")
print("Final shapefile:", final_shp)


Copy to GDB (avoids locks)...
Overwriting Rech_m_d from HK lake flag...
✅ Lake polys (HK=100): 30
✅ Land polys:          44
Export new shapefile...

DONE ✅
Output: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\surfacegeology_Kh_clip2km_Rech_RECHFIX.shp


## Fix Rech_m_d using HK lake flag (recommended)

In [ ]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------
in_shp = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\surfacegeology_Kh_clip2km_Rech.shp"

# lake flag comes from HK fields already being set to 100 in lakes
HK_FIELDS = ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]
RECH_FIELD = "Rech_m_d"

# if HK is ~100 (float safe), treat as lake
HK_LAKE_THRESHOLD = 99.5

# optional: clear these (set NULL) on lake polygons
CLEAR_ON_LAKE = ["Upper_ms", "Middle_ms", "Lower_ms"]   # set [] to disable

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
out_dir = os.path.dirname(in_shp)
gdb = os.path.join(out_dir, "HK_rech_fix.gdb")
if not arcpy.Exists(gdb):
    arcpy.management.CreateFileGDB(out_dir, os.path.basename(gdb))

work_fc = os.path.join(gdb, "hk_work_rechfix")
out_shp = os.path.join(out_dir, "surfacegeology_Kh_clip2km_Rech_RECHFIX.shp")

def delete_if_exists(p):
    if p and arcpy.Exists(p):
        arcpy.management.Delete(p)

def ensure_field(fc, name, ftype):
    existing = {f.name for f in arcpy.ListFields(fc)}
    if name not in existing:
        arcpy.management.AddField(fc, name, ftype)

print("Copy to GDB (avoids locks)...")
delete_if_exists(work_fc)
arcpy.management.CopyFeatures(in_shp, work_fc)

# ensure Rech exists and is editable
ensure_field(work_fc, RECH_FIELD, "DOUBLE")

fields = {f.name for f in arcpy.ListFields(work_fc)}
for f in HK_FIELDS:
    if f not in fields:
        raise RuntimeError(f"Missing HK field {f} in {work_fc}. Fields: {sorted(fields)}")

# keep only clear fields that exist
clear_fields = [f for f in CLEAR_ON_LAKE if f in fields]

print("Overwriting Rech_m_d from HK lake flag...")
n_lake = 0
n_land = 0

update_fields = HK_FIELDS + [RECH_FIELD] + clear_fields

with arcpy.da.UpdateCursor(work_fc, update_fields) as cur:
    for row in cur:
        up, mid, low = row[0], row[1], row[2]

        def is100(v):
            try:
                return (v is not None) and (float(v) >= HK_LAKE_THRESHOLD)
            except:
                return False

        is_lake = is100(up) and is100(mid) and is100(low)

        # set Rech
        row[3] = 100.0 if is_lake else 0.0

        # clear other fields for lake polys (optional)
        if is_lake and clear_fields:
            for j in range(4, len(update_fields)):
                row[j] = None

        cur.updateRow(row)

        if is_lake:
            n_lake += 1
        else:
            n_land += 1

print(f"✅ Lake polys (HK=100): {n_lake}")
print(f"✅ Land polys:          {n_land}")

print("Export new shapefile...")
delete_if_exists(out_shp)
arcpy.conversion.FeatureClassToFeatureClass(work_fc, out_dir, os.path.basename(out_shp))

print("\nDONE ✅")
print("Output:", out_shp)


## Create a model bottom raster from the LowKh_m_d column 

In [25]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------
hk_poly = r"\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\surfacegeology_Kh_clip2km_Rech_RECHFIX.shp"
value_field = "LowKh_m_d"

# Optional (RECOMMENDED): a raster that defines your model grid (cellsize/snap/extent)
# Example: your 30m mask raster
template_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif"
# If you DON'T want to use a template raster, set template_raster = None and set cell_size below.

# If no template raster:
cell_size = 30  # meters (only used when template_raster=None)

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom"
os.makedirs(out_dir, exist_ok=True)

out_raster = os.path.join(out_dir, "modelbottom.tif")

# ------------------------------------------------------------
# ENV: align to template if provided
# ------------------------------------------------------------
if template_raster and arcpy.Exists(template_raster):
    arcpy.env.snapRaster = template_raster
    arcpy.env.cellSize = template_raster
    arcpy.env.extent = template_raster
    arcpy.env.outputCoordinateSystem = arcpy.Describe(template_raster).spatialReference
    print("✅ Using template raster for snap/cellsize/extent/CRS:", template_raster)
else:
    arcpy.env.snapRaster = None
    arcpy.env.cellSize = cell_size
    arcpy.env.extent = hk_poly
    arcpy.env.outputCoordinateSystem = arcpy.Describe(hk_poly).spatialReference
    print("⚠️ No template raster used. Using cell_size =", cell_size)

# ------------------------------------------------------------
# CHECK FIELD EXISTS
# ------------------------------------------------------------
fields = [f.name for f in arcpy.ListFields(hk_poly)]
if value_field not in fields:
    raise RuntimeError(f"Field '{value_field}' not found. Fields: {fields}")

# ------------------------------------------------------------
# POLYGON -> RASTER
# ------------------------------------------------------------
if arcpy.Exists(out_raster):
    arcpy.management.Delete(out_raster)

# Use "CELL_CENTER" assignment; use MAXIMUM_AREA if you prefer dominance by area
arcpy.conversion.PolygonToRaster(
    in_features=hk_poly,
    value_field=value_field,
    out_rasterdataset=out_raster,
    cell_assignment="CELL_CENTER",
    priority_field="NONE",
    cellsize=arcpy.env.cellSize
)

print("\nDONE ✅")
print("Output raster:", out_raster)


✅ Using template raster for snap/cellsize/extent/CRS: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif

DONE ✅
Output raster: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Bottom\modelbottom.tif


## Create a multiband raster from the fields 
fields = ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]  

In [26]:
import os
import arcpy

arcpy.env.overwriteOutput = True

# ------------------------------------------------------------
# INPUTS
# ------------------------------------------------------------
hk_poly = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\surfacegeology_Kh_clip2km_Rech_RECHFIX.shp"

fields = ["UPKh_m_d", "MidKh_m_d", "LowKh_m_d"]  # band order

# Recommended: template raster to match your model grid
template_raster = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif"
# If you don't want to use a template raster, set template_raster=None and set cell_size below.
cell_size = 30

# ------------------------------------------------------------
# OUTPUTS
# ------------------------------------------------------------
out_dir = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK"
os.makedirs(out_dir, exist_ok=True)

# Multiband output (GeoTIFF)
out_multiband = os.path.join(out_dir, "HK_multiband.tif")

# Temp single-band rasters
tmp_dir = os.path.join(out_dir, "_tmp_hk_bands")
os.makedirs(tmp_dir, exist_ok=True)

# ------------------------------------------------------------
# ENV ALIGNMENT
# ------------------------------------------------------------
if template_raster and arcpy.Exists(template_raster):
    arcpy.env.snapRaster = template_raster
    arcpy.env.cellSize = template_raster
    arcpy.env.extent = template_raster
    arcpy.env.outputCoordinateSystem = arcpy.Describe(template_raster).spatialReference
    print("✅ Using template raster for snap/cellsize/extent/CRS:", template_raster)
else:
    arcpy.env.snapRaster = None
    arcpy.env.cellSize = cell_size
    arcpy.env.extent = hk_poly
    arcpy.env.outputCoordinateSystem = arcpy.Describe(hk_poly).spatialReference
    print("⚠️ No template raster used. Using cell_size =", cell_size)

# ------------------------------------------------------------
# CHECK FIELDS
# ------------------------------------------------------------
poly_fields = {f.name for f in arcpy.ListFields(hk_poly)}
missing = [f for f in fields if f not in poly_fields]
if missing:
    raise RuntimeError(f"Missing fields in hk_poly: {missing}\nFields found: {sorted(poly_fields)}")

# ------------------------------------------------------------
# 1) Polygon -> Raster for each field
# ------------------------------------------------------------
band_rasters = []
for f in fields:
    out_ras = os.path.join(tmp_dir, f"{f}.tif")
    if arcpy.Exists(out_ras):
        arcpy.management.Delete(out_ras)

    arcpy.conversion.PolygonToRaster(
        in_features=hk_poly,
        value_field=f,
        out_rasterdataset=out_ras,
        cell_assignment="CELL_CENTER",
        priority_field="NONE",
        cellsize=arcpy.env.cellSize
    )
    band_rasters.append(out_ras)
    print(f"✅ Created band raster for {f}: {out_ras}")

# ------------------------------------------------------------
# 2) Composite bands into a multiband raster
# ------------------------------------------------------------
if arcpy.Exists(out_multiband):
    arcpy.management.Delete(out_multiband)

arcpy.management.CompositeBands(band_rasters, out_multiband)

print("\nDONE ✅")
print("Multiband HK raster:", out_multiband)
print("Band order:", fields)


✅ Using template raster for snap/cellsize/extent/CRS: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Costantheads\domain_water_mask_30m_buff20000m.tif
✅ Created band raster for UPKh_m_d: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\_tmp_hk_bands\UPKh_m_d.tif
✅ Created band raster for MidKh_m_d: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\_tmp_hk_bands\MidKh_m_d.tif
✅ Created band raster for LowKh_m_d: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\_tmp_hk_bands\LowKh_m_d.tif

DONE ✅
Multiband HK raster: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\HK\HK_multiband.tif
Band order: ['UPKh_m_d', 'MidKh_m_d', 'LowKh_m_d']


# convert Ibound to tif

In [24]:
Ibound = r'S:\Projects\Active\GLB_LHM\LHM_inputs\Boundary\extendedBdry_jan26_adk.shp'
Ibound_ras = r'D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extendedBdry_jan26_adk.tif'
dem = r"D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\DEM\DEM_merged_3174_30m.tif"

import arcpy
arcpy.env.overwriteOutput = True
arcpy.env.snapRaster = dem
arcpy.env.cellSize = dem

arcpy.conversion.PolygonToRaster(
    in_features=Ibound,
    value_field="FID",
    out_rasterdataset=Ibound_ras,
    cell_assignment="MAXIMUM_AREA",
    priority_field="NONE",
    cellsize=dem
)
print("✅ Rasterized Ibound to:", Ibound_ras)

✅ Rasterized Ibound to: D:\Users\abolmaal\modelling\Modflow\Prep\GreatLakes\model_Layers\Ibound\extendedBdry_jan26_adk.tif
